# NASA POWER environmental data fix
This cell uses the NASA POWER API with robust year-by-year downloading and correct JSON parsing.

In [ ]:
import time, requests, numpy as np, pandas as pd

# Central India representative agricultural location
LAT, LON = 22.9734, 78.6569
start_year, end_year = int(d['year'].min()), int(d['year'].max())
parameters = 'T2M,PRECTOTCORR,RH2M,ALLSKY_SFC_SW_DWN'

def download_power_year(year):
    url = 'https://power.larc.nasa.gov/api/temporal/daily/point'
    query = {
        'parameters': parameters,
        'community': 'AG',
        'longitude': LON,
        'latitude': LAT,
        'start': f'{year}0101',
        'end': f'{year}1231',
        'format': 'JSON'
    }
    response = requests.get(url, params=query, timeout=180)
    response.raise_for_status()
    payload = response.json()
    if 'properties' not in payload or 'parameter' not in payload['properties']:
        raise ValueError(f'Unexpected NASA POWER response for {year}: {payload}')
    parameter_data = payload['properties']['parameter']
    frame = pd.DataFrame(parameter_data)
    frame.index = pd.to_datetime(frame.index.astype(str), format='%Y%m%d', errors='coerce')
    frame = frame[frame.index.notna()].replace(-999, np.nan)
    return frame

weather_frames = []
failed_years = []
for year in range(start_year, end_year + 1):
    try:
        weather_frames.append(download_power_year(year))
        print(f'Downloaded NASA POWER data for {year}')
        time.sleep(0.2)
    except Exception as e:
        print(f'Warning: could not download {year}: {e}')
        failed_years.append(year)

if not weather_frames:
    raise RuntimeError('NASA POWER download failed for all years. Check internet access and try again.')

w = pd.concat(weather_frames).sort_index()
w = w.apply(pd.to_numeric, errors='coerce')

# Aggregate daily real observations into annual environmental features
a = (w.resample('YE')
       .agg({
           'T2M': 'mean',
           'PRECTOTCORR': 'sum',
           'RH2M': 'mean',
           'ALLSKY_SFC_SW_DWN': 'mean'
       })
       .reset_index())
a['year'] = a['index'].dt.year
a = a.drop(columns=['index'])
a = a.dropna(how='all', subset=['T2M','PRECTOTCORR','RH2M','ALLSKY_SFC_SW_DWN'])

print('Environmental dataset shape:', a.shape)
print('Failed years:', failed_years if failed_years else 'None')
display(a.head())
a.to_csv('nasa_power_environmental_data.csv', index=False)
